# 00 — Setup (Colab GPU)

**OPG-Live** — Day 1-3 setup. Edit di VS Code, **Run All di Colab Web**.

1. Runtime → Change runtime type → **T4 GPU** (free) atau L4/A100 (Pro)
2. **Runtime → Run all**

Notebook ini: mount Drive → cek GPU → install deps → clone repo → setup folder Drive → verify DENTEX → test SAM ViT-H.

## Cell 1 — Mount Google Drive (data & checkpoints persist di sini)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/opg-live'
print('Drive mounted. Project root:', DRIVE_ROOT)

## Cell 2 — Cek GPU (harus True; GTX 1650 laptop tidak cukup untuk ViT-H)

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM           : {total/1e9:.1f} GB total, {free/1e9:.1f} GB free')
    assert total/1e9 > 6, 'VRAM < 6GB — ganti ke T4/L4 untuk SAM ViT-H'
else:
    raise RuntimeError('GPU OFF → Runtime > Change runtime type > T4 GPU')

## Cell 3 — Install dependencies (~3-5 menit)

In [ ]:
%pip install -q segment-anything pycocotools pdfplumber \
    sentence-transformers openai numpy opencv-python-headless scipy \
    scikit-learn statsmodels matplotlib
print('Core deps installed.')

In [ ]:
# detectron2 (untuk HierarchicalDet Stage 1) — install dari source
%pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print('detectron2 installed.')

## Cell 4 — Clone / pull repo kode dari GitHub

In [ ]:
import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
else:
    !cd {REPO} && git pull
%cd {REPO}
!ls

## Cell 5 — Setup folder Drive (data / checkpoints / outputs)

In [ ]:
import os
for sub in ['data/dentex', 'data/kb', 'checkpoints', 'outputs/masks', 'outputs/reports', 'outputs/metrics']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

# Symlink: kode lihat ./data ./checkpoints ./outputs tapi fisik di Drive
for link in ['data', 'checkpoints', 'outputs']:
    if os.path.islink(link) or os.path.exists(link):
        !rm -rf {link}
    os.symlink(f'{DRIVE_ROOT}/{link}', link)
print('Folder Drive siap + symlink ke repo:')
!ls -la data checkpoints outputs

## Cell 6 — Verify DENTEX dataset

**Sekali saja:** zip `JSON DATASET DENTEX` di laptop → upload ke `MyDrive/opg-live/data/dentex/`. Cell ini cek class distribution (harus Caries 2189, Impacted 604, Periapical 158, Deep Caries 578).

In [ ]:
import glob, json
from collections import Counter

anns = glob.glob(f'{DRIVE_ROOT}/data/dentex/**/*.json', recursive=True)
print('JSON ditemukan:', len(anns))
if not anns:
    print('⚠️  Belum ada data. Upload zip DENTEX ke MyDrive/opg-live/data/dentex/ lalu unzip.')
else:
    # cari file dengan annotations + categories (disease)
    for f in anns:
        d = json.load(open(f))
        if isinstance(d, dict) and 'annotations' in d:
            cats = {c['id']: c['name'] for c in d.get('categories', [])}
            seg = [a for a in d['annotations'] if a.get('segmentation')]
            dist = Counter(a['category_id'] for a in seg)
            print('File           :', f.split('/')[-1])
            print('Polygon (seg)  :', len(seg))
            print('Distribution   :', {cats.get(k, k): v for k, v in dist.items()})
            break

## Cell 7 — Download + test SAM ViT-H (~2.5 GB, sekali; cache ke Drive)

In [ ]:
import os, urllib.request
CKPT = f'{DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth'
URL  = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
if not os.path.exists(CKPT):
    print('Downloading SAM ViT-H ke Drive...')
    urllib.request.urlretrieve(URL, CKPT)
print('SAM checkpoint:', CKPT, f'({os.path.getsize(CKPT)/1e9:.2f} GB)')

In [ ]:
from segment_anything import sam_model_registry, SamPredictor
import torch
sam = sam_model_registry['vit_h'](checkpoint=CKPT).to('cuda')
predictor = SamPredictor(sam)
n = sum(p.numel() for p in sam.parameters())
print(f'✅ SAM ViT-H loaded di GPU — {n/1e6:.0f}M params. Stage 2 siap.')

## Cell 8 — OpenAI key (GPT-4o, Stage 3). Simpan key di Colab Secrets (kunci kiri 🔑), JANGAN hardcode.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    from openai import OpenAI
    OpenAI()  # validasi key terbaca
    print('✅ OPENAI_API_KEY OK — GPT-4o siap dipanggil.')
except Exception as e:
    print('⚠️  Set dulu: Colab kiri 🔑 Secrets → OPENAI_API_KEY → toggle Notebook access. Detail:', e)

## ✅ Checklist Phase 0

- [ ] Drive mounted (Cell 1)
- [ ] GPU aktif, VRAM > 6GB (Cell 2)
- [ ] Deps + detectron2 installed (Cell 3)
- [ ] Repo cloned/pulled (Cell 4)
- [ ] Folder Drive + symlink (Cell 5)
- [ ] DENTEX class distribution confirmed (Cell 6)
- [ ] SAM ViT-H loaded di GPU (Cell 7)
- [ ] OPENAI_API_KEY OK (Cell 8)

**Next:** `notebooks/01_sam_adapter_train.ipynb` (Phase A — train Medical SAM Adapter).